In [1]:
import boto3
import pandas as pd
import sagemaker
from sagemaker import get_execution_role
from sklearn.cluster import KMeans
import joblib
import os
from datetime import datetime

# Config
s3_input_path = "s3://swo-ngoctran-public/mlops/pipelines/01_preprocessing_kmeans/2024-01/30/23/sample_output1_DWA_processed.parquet"
s3_output_model_path = "s3://swo-ngoctran-public/mlops/pipelines/02_training_kmeans/"
model_name = "kmeans-speed-model"
endpoint_name = "kmeans-speed-endpoint"
local_model_path = "kmeans_model.pkl"

region = sagemaker.Session().boto_region_name
role = get_execution_role()
session = sagemaker.Session()
bucket = "swo-ngoctran-public"

# Step 1: Load data
df = pd.read_parquet(s3_input_path)
print("Original DataFrame shape:", df.shape)

# Step 2: Filter only the "value" column
X = df[["value"]].dropna()
print("Training data shape:", X.shape)

# Step 3: Train KMeans model
kmeans = KMeans(n_clusters=5, random_state=42)
kmeans.fit(X)

# Step 4: Assign cluster back to original data
df = df.copy()
df["speed_cluster"] = kmeans.predict(X)

# Step 5: Save model locally and upload to S3
joblib.dump(kmeans, local_model_path)

timestamp = datetime.utcnow().strftime("%Y-%m-%d-%H%M%S")
s3_model_key = f"mlops/pipelines/02_training_kmeans/kmeans_model_{timestamp}.pkl"

s3 = boto3.client("s3")
s3.upload_file(local_model_path, bucket, s3_model_key)
print("Model uploaded to:", f"s3://{bucket}/{s3_model_key}")


sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
Original DataFrame shape: (1878424, 10)
Training data shape: (1878424, 1)


/tmp/ipykernel_7966/3508128257.py:41: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime("%Y-%m-%d-%H%M%S")


Model uploaded to: s3://swo-ngoctran-public/mlops/pipelines/02_training_kmeans/kmeans_model_2025-07-07-061813.pkl


In [4]:
import tarfile

model_dir = "model"
os.makedirs(model_dir, exist_ok=True)
joblib.dump(kmeans, os.path.join(model_dir, "kmeans_model.pkl"))

with tarfile.open("model.tar.gz", "w:gz") as tar:
    tar.add(model_dir, arcname=".")


In [5]:
s3_model_key = f"mlops/pipelines/02_training_kmeans/kmeans_model_{timestamp}.tar.gz"
s3.upload_file("model.tar.gz", bucket, s3_model_key)


In [6]:
model_data = f"s3://{bucket}/{s3_model_key}"


In [7]:
def model_fn(model_dir):
    return joblib.load(os.path.join(model_dir, "kmeans_model.pkl"))


In [9]:
from sagemaker.sklearn.model import SKLearnModel

sk_model = SKLearnModel(
    model_data=f"s3://{bucket}/{s3_model_key}",
    role=role,
    entry_point="inference.py",  # ensure this file exists and correct
    framework_version="0.23-1",
    sagemaker_session=session
)

import uuid

endpoint_name = f"kmeans-speed-endpoint-{uuid.uuid4().hex[:6]}"

predictor = sk_model.deploy(
    instance_type="ml.m5.large",
    initial_instance_count=1,
    endpoint_name=endpoint_name
)
print(f"Endpoint deployed: {endpoint_name}")



----------------------------------------------*

Please check the troubleshooting guide for common errors: https://docs.aws.amazon.com/sagemaker/latest/dg/sagemaker-python-sdk-troubleshooting.html#sagemaker-python-sdk-troubleshooting-create-endpoint


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:15                                                                                   │
│                                                                                                  │
│   12                                                                                             │
│   13 endpoint_name = f"kmeans-speed-endpoint-{uuid.uuid4().hex[:6]}"                             │
│   14                                                                                             │
│ ❱ 15 predictor = sk_model.deploy(                                                                │
│   16 │   instance_type="ml.m5.large",                                                            │
│   17 │   initial_instance_count=1,                                                               │
│   18 │   endpoint_name=endpoint_name                                                             │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/model.py:1786 in deploy                        │
│                                                                                                  │
│   1783 │   │   │   if is_explainer_enabled:                                                      │
│   1784 │   │   │   │   explainer_config_dict = explainer_config._to_request_dict()               │
│   1785 │   │   │                                                                                 │
│ ❱ 1786 │   │   │   self.sagemaker_session.endpoint_from_production_variants(                     │
│   1787 │   │   │   │   name=self.endpoint_name,                                                  │
│   1788 │   │   │   │   production_variants=[production_variant],                                 │
│   1789 │   │   │   │   tags=tags,                                                                │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/session.py:5940 in                             │
│ endpoint_from_production_variants                                                                │
│                                                                                                  │
│   5937 │   │   logger.info("Creating endpoint-config with name %s", name)                        │
│   5938 │   │   self.sagemaker_client.create_endpoint_config(**config_options)                    │
│   5939 │   │                                                                                     │
│ ❱ 5940 │   │   return self.create_endpoint(                                                      │
│   5941 │   │   │   endpoint_name=name,                                                           │
│   5942 │   │   │   config_name=name,                                                             │
│   5943 │   │   │   tags=endpoint_tags,                                                           │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/session.py:4785 in create_endpoint             │
│                                                                                                  │
│   4782 │   │   │   logger.error(                                                                 │
│   4783 │   │   │   │   "Please check the troubleshooting guide for common errors: %s", troubles  │
│   4784 │   │   │   )                                                                             │
│ ❱ 4785 │   │   │   raise e                                                                       │
│   4786 │                                                                                         │
│   4787 │   def endpoint_in_service_or_not(self, endpoint_na

In [ ]:
import json

dummy_data = pd.DataFrame({
    "value": [10.0, 200.0, 320.0, 500.0]
})

response = predictor.predict(dummy_data.to_dict(orient="records"))
print("Prediction:", response)
